# Data Modeling

The goal of this file is to examine the master census metrics JSON file for the CCSVI Dashboard and understand how to transition the existing data into a format usable by an LLM.

## Updates

- master JSON file: `public/data/metrics/census_metrics_by_block_group.json`.
- flattened the nested JSON into a long-form Pandas DataFrame.

## Next Steps
- cleaning and transforming metrics, and preparing the dataset for analysis or LLM integration.

In [43]:
import json
import pandas as pd

In [50]:
# Loading the master .json file
master_file = "../public/data/metrics/census_metrics_by_block_group.json"
with open(master_file, "r") as f:
    data = json.load(f)

print("Type of raw_data:", type(data))
print("Number of geographic units:", len(data))
print("Sample keys:", list(data.keys())[:5])

Type of raw_data: <class 'dict'>
Number of geographic units: 1158
Sample keys: ['5003', '5004', '5008', '5009', '5011']


In [53]:
# Flattening the .json into Long-Format
rows = []

for geoid, area_data in data.items():
    base_info = {
        "geoid": geoid,
        "type": area_data.get("type"),
        "name": area_data.get("name"),
        "population": area_data.get("population")
    }
    
    metrics_block = area_data.get("metrics", {})
    
    for dataset_name, metrics in metrics_block.items():
        for metric_name, values in metrics.items():
            row = base_info.copy()
            row["dataset"] = dataset_name
            row["metric"] = metric_name
            row["absolute"] = values.get("absolute")
            row["proportion"] = values.get("proportion")
            rows.append(row)

df_long = pd.DataFrame(rows)

In [55]:
# Check
print(df_long.head())
print("Shape:", df_long.shape)
print("Unique metrics:", df_long["metric"].nunique())
print("Unique geographic units:", df_long["geoid"].nunique())

  geoid               type                                           name  \
0  5003  hawaiian_homeland  Anahola (Agricultural) Hawaiian Home Land, HI   
1  5003  hawaiian_homeland  Anahola (Agricultural) Hawaiian Home Land, HI   
2  5003  hawaiian_homeland  Anahola (Agricultural) Hawaiian Home Land, HI   
3  5003  hawaiian_homeland  Anahola (Agricultural) Hawaiian Home Land, HI   
4  5003  hawaiian_homeland  Anahola (Agricultural) Hawaiian Home Land, HI   

   population                         dataset                       metric  \
0       257.0  2022_census_hawaiian_homelands     Total Population Under 5   
1       257.0  2022_census_hawaiian_homelands    Total Population Under 18   
2       257.0  2022_census_hawaiian_homelands     Total Population Over 65   
3       257.0  2022_census_hawaiian_homelands    Total population SEX Male   
4       257.0  2022_census_hawaiian_homelands  Total population SEX Female   

   absolute  proportion  
0      16.0    0.062257  
1      39.0    0

### Data Structure

The master JSON is structured as follows:

- Top-level keys: `geoid` for each geographic unit.  
- Each unit contains:
  - `type` → type of area (e.g., "hawaiian_homeland")
  - `name` → full name of the unit
  - `population` → total population
  - `metrics` → nested dictionary with:
    - dataset name (e.g., "2022_census_hawaiian_homelands")
    - metric name → dictionary with `absolute` and `proportion`

Long-format: one row per metric per geographic unit.
Wide-format: one row per geographic unit, metrics as columns → easier for LLM ingestion.

In [60]:
# Pivot long DataFrame to Wide-format
df_wide = df_long.pivot_table(
    index=["geoid", "type", "name", "population"],
    columns="metric",
    values=["absolute", "proportion"]
).reset_index()

df_wide.columns = ['_'.join(filter(None, col)).strip() for col in df_wide.columns.values]

print(df_wide.head())
print("Shape:", df_wide.shape)

          geoid         type  \
0  150010201001  block_group   
1  150010201002  block_group   
2  150010201003  block_group   
3  150010201004  block_group   
4  150010202021  block_group   

                                                name  population  \
0  Block Group 1; Census Tract 201; Hawaii County...      1826.0   
1  Block Group 2; Census Tract 201; Hawaii County...      1044.0   
2  Block Group 3; Census Tract 201; Hawaii County...      1350.0   
3  Block Group 4; Census Tract 201; Hawaii County...      1213.0   
4  Block Group 1; Census Tract 202.02; Hawaii Cou...       888.0   

   absolute_American Indian and Alaska Native alone  absolute_Asian alone  \
0                                              14.0                 450.0   
1                                               0.0                 125.0   
2                                               0.0                 272.0   
3                                               0.0                 310.0   
4            

### LLM Use Cases

- Summarize population and income patterns across units.  
- Identify correlations between demographics, race, age, and income.  
- Generate insights or reports, e.g.:  
  - "Which Hawaiian Home Land has the highest median income?"  
  - "Which units have more than 50% of population under 18?"  